# Aufgaben 6 – Modell 2: Cross-Validation mit k-Fold
**Kurs:** NKDS – Machine Learning | **Dozentin:** Prof. Dr. Jennifer Schoch, DHBW Karlsruhe  
**Datensatz:** LLM - Detect AI Generated Text | **Gruppe:** [Namen eintragen] | **Datum:** [Datum]

---

## Theoretischer Hintergrund

**Cross-Validation (Kreuzvalidierung)** ist keine eigene Modellklasse, sondern eine **Evaluierungsmethode**, die robustere Schätzungen der Modellgüte liefert als ein einfacher Train/Val-Split.

**k-Fold Cross-Validation:**
1. Der Datensatz wird in **k gleich große Teile (Folds)** aufgeteilt.
2. Das Modell wird **k-mal** trainiert – jedes Mal ist ein anderes Fold das Validierungsset, der Rest ist Trainingsdaten.
3. Die k Ergebnisse werden gemittelt → stabilere Schätzung, weniger abhängig vom Zufall des Splits.

**Vorteile gegenüber einfachem Train/Val-Split:**
- Jedes Sample wird genau einmal zur Validierung verwendet
- Geringere Varianz der Schätzung
- Besonders nützlich bei kleineren Datensätzen

> In diesem Notebook wenden wir k-Fold Cross-Validation auf mehrere Klassifikationsmodelle an und vergleichen ihre CV-Scores.


## 0. Setup & Daten laden

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

df = pd.read_csv("train_essays.csv").dropna(subset=["text"]).drop_duplicates(subset=["text"])
print(f"Datensatz: {df.shape[0]} Zeilen, {df.shape[1]} Spalten")
print(f"Klassenverteilung:\n{df['generated'].value_counts()}")


## 1a. Train / Test Split

Bei Cross-Validation brauchen wir nur einen Train/Test-Split – die Validierung übernimmt das k-Fold-Verfahren automatisch.

In [ ]:
X = df["text"]
y = df["generated"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Trainingsdaten (für CV): {len(X_train)} Samples")
print(f"Testdaten (zurückgehalten): {len(X_test)} Samples")


## 1b. Feature-Pipeline aufbauen

Damit kein Data Leakage entsteht, muss der TF-IDF-Vectorizer **innerhalb** jedes CV-Folds gefittet werden. Das erreichen wir mit einer **sklearn Pipeline**.

In [ ]:
# Pipelines: TF-IDF + Modell in einem Objekt
# So wird der Vectorizer bei jeder CV-Iteration nur auf den Trainingsteil gefittet

pipelines = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, sublinear_tf=True)),
        ("clf",   LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, sublinear_tf=True)),
        ("clf",   MultinomialNB())
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, sublinear_tf=True)),
        ("clf",   LinearSVC(max_iter=2000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, sublinear_tf=True)),
        ("clf",   DecisionTreeClassifier(max_depth=10, random_state=42))
    ]),
}

print("Pipelines definiert:")
for name in pipelines:
    print(f"  • {name}")


## 1b–1c. k-Fold Cross-Validation durchführen

Wir verwenden **Stratified k-Fold** (k=5), das die Klassenverteilung in jedem Fold beibehält – wichtig bei potenziell unausgeglichenen Klassen.


In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, pipeline in pipelines.items():
    scores = cross_val_score(pipeline, X_train, y_train,
                             cv=cv_strategy, scoring="accuracy", n_jobs=-1)
    cv_results.append({
        "Modell":        name,
        "CV Mean Acc":   round(scores.mean(), 4),
        "CV Std Acc":    round(scores.std(), 4),
        "Fold Scores":   [round(s, 4) for s in scores],
    })
    print(f"{name:25s} | CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")

cv_df = pd.DataFrame(cv_results).sort_values("CV Mean Acc", ascending=False)
cv_df[["Modell", "CV Mean Acc", "CV Std Acc", "Fold Scores"]]


## 1f. Visualisierung – CV-Ergebnisse im Vergleich

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) Balkenplot: CV Mean ± Std
models_sorted = cv_df["Modell"].tolist()
means  = cv_df["CV Mean Acc"].tolist()
stds   = cv_df["CV Std Acc"].tolist()
colors = ["steelblue", "mediumseagreen", "tomato", "goldenrod"]

axes[0].barh(models_sorted, means, xerr=stds, align="center",
             color=colors[:len(models_sorted)], alpha=0.8, capsize=5, edgecolor="black")
axes[0].set_xlabel("CV Accuracy")
axes[0].set_title("5-Fold CV Accuracy (Mittelwert ± Std)")
axes[0].set_xlim(0.5, 1.05)
axes[0].axvline(x=max(means), color="gray", linestyle=":", alpha=0.5)
axes[0].grid(axis="x", alpha=0.3)

# (b) Boxplot: Verteilung der Fold-Scores
fold_data  = [row["Fold Scores"] for _, row in cv_df.iterrows()]
bp = axes[1].boxplot(fold_data, labels=models_sorted, patch_artist=True,
                     medianprops={"color": "black", "linewidth": 2})
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel("Accuracy je Fold")
axes[1].set_title("Streuung der Fold-Scores (5-Fold CV)")
axes[1].set_xticklabels(models_sorted, rotation=15, ha="right")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()


**Beobachtung:**  
- Ein **niedriges Std** zeigt, dass das Modell stabil über alle Folds performt.  
- Ein **hohes Std** deutet auf Sensitivität gegenüber dem gewählten Split hin.  
- [Eigene Beobachtungen eintragen: Welches Modell ist am stabilsten? Welches am besten?]


## Vertiefung: Lernkurve (Train-Size vs. Val-Accuracy)

In [ ]:
from sklearn.model_selection import learning_curve

# Lernkurve für das beste Modell
best_pipeline_name = cv_df.iloc[0]["Modell"]
best_pipeline      = pipelines[best_pipeline_name]

train_sizes, train_scores, val_scores = learning_curve(
    best_pipeline, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    train_sizes=np.linspace(0.1, 1.0, 8),
    scoring="accuracy", n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", color="steelblue", label="Train Accuracy")
ax.fill_between(train_sizes,
                train_scores.mean(axis=1) - train_scores.std(axis=1),
                train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15, color="steelblue")
ax.plot(train_sizes, val_scores.mean(axis=1), "s--", color="tomato", label="CV Val Accuracy")
ax.fill_between(train_sizes,
                val_scores.mean(axis=1) - val_scores.std(axis=1),
                val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.15, color="tomato")

ax.set_xlabel("Anzahl Trainingssamples")
ax.set_ylabel("Accuracy")
ax.set_title(f"Lernkurve – {best_pipeline_name}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Interpretation der Lernkurve:**  
- Nähern sich Train- und Val-Kurve an → das Modell generalisiert gut.  
- Großer Abstand → Overfitting (mehr Daten oder Regularisierung helfen).  
- Val-Kurve steigt noch → mehr Daten würden die Performance weiter verbessern.  

[Eigene Interpretation eintragen]


## 1d. Bestes Modell auf Testdaten evaluieren

In [ ]:
best_pipeline_name = cv_df.iloc[0]["Modell"]
best_pipeline      = pipelines[best_pipeline_name]
print(f"Bestes Modell (nach CV): {best_pipeline_name}")

# Finales Training auf allen Trainingsdaten
best_pipeline.fit(X_train, y_train)
y_pred_test = best_pipeline.predict(X_test)
test_acc    = accuracy_score(y_test, y_pred_test)

print(f"Test Accuracy: {test_acc:.4f}")
print("\nKlassifikationsbericht:")
print(classification_report(y_test, y_pred_test, target_names=["Mensch (0)", "KI (1)"]))


In [ ]:
cm = confusion_matrix(y_test, y_pred_test)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Mensch (0)", "KI (1)"]).plot(cmap="Blues")
plt.title(f"Konfusionsmatrix – {best_pipeline_name}")
plt.tight_layout()
plt.show()


## 1e. Stichprobe: Vorhersage vs. echter Wert

In [ ]:
sample_idx   = X_test.sample(10, random_state=7).index
sample_texts = X_test.loc[sample_idx]
sample_true  = y_test.loc[sample_idx]
sample_pred  = best_pipeline.predict(sample_texts)

pd.DataFrame({
    "Text (Ausschnitt)": [t[:80] + "..." for t in sample_texts],
    "Wahrer Wert":       sample_true.values,
    "Vorhersage":        sample_pred,
    "Korrekt?":          ["✅" if t==p else "❌" for t, p in zip(sample_true.values, sample_pred)]
})


## 1g. Erkenntnisse & Bewertung

### Zusammenfassung

| Modell | CV Accuracy (Ø) | CV Std | Test Accuracy |
|--------|----------------|--------|---------------|
| [eintragen] | | | |

### Warum ist Cross-Validation besser als ein einfacher Split?
- Ein einzelner Train/Val-Split kann je nach Zufall der Aufteilung stark variieren.
- 5-Fold CV nutzt die Daten effizienter – jeder Datenpunkt wird einmal validiert.
- Das Std der Fold-Scores zeigt, wie stabil das Modell wirklich ist.

### Stärken & Schwächen von k-Fold CV
- ✅ Robustere Modellschätzung, weniger zufällig
- ✅ Gibt Aufschluss über Modellstabilität (Std)
- ❌ k-facher Rechenaufwand gegenüber einfachem Split
- ❌ Bei sehr großen Datensätzen oft impraktikabel

### Was haben wir erwartet / Was hat überrascht?
[Eigene Reflexion der Gruppe eintragen]
